# 3b. Score, second server (optional): the same block list, walked backwards

Run this on a SECOND CPU high-RAM server while `03_score` runs on the first. The two start from opposite ends of
`BLOCKS` and meet in the middle. Before a server starts a block it leaves a small claim file in persistent storage;
the other server skips a claimed block. A claim older than 25 minutes (a server that died) is taken over, and if
both ever grab the same block in the same moment, one block of work is wasted and the numbers are identical.

Keep cell 1 identical to `03_score`; only `REVERSE` differs. Not needed at all: `03_score` alone does everything.

**About the saved output below.** It is older than the last round of speed-ups; see the note in `03_score`. The scores do not depend on it.

**Next:** `04_report`.

In [1]:
# --- 1. Configuration ---
PERSIST_MODE = "drive"
DRIVE_ROOT = "/content/drive/MyDrive/vggt-omega-aura-benchmark"   # where predictions, ground truth and results live.
# Work already saved there is skipped. To run EVERYTHING again from the images up, name an empty folder here,
# the same one in every notebook of the run. The Hugging Face token is still found in the usual folder's .env.
RUN_TAG = "phase7_front_medium"       # the folder of this run in persistent storage. A name from the time of the work:
                                      # every notebook of the run must use the same one, the results live under it
CAMERA = "front_medium"
MODELS = ["vggt_omega_512", "vggt_1b"]
BLOCKS = [("train", 11), ("train", 12), ("train", 13),              # dark and wet: every scene is both (two recordings of one evening)
          ("train", 92), ("train", 82), ("train", 83), ("train", 90),   # motorway-rich
          ("train", 5), ("train", 6), ("train", 9),                 # wet in DAYLIGHT, to tell rain from darkness
          ("val", 0), ("val", 1), ("val", 2), ("val", 10), ("val", 12)]   # the validation blocks
# Chosen from the census for rare conditions, not at random. Finished blocks are skipped, so the list can grow.
# ("val", 11) is the development block (notebooks/development). The test blocks have their own notebooks, 05 to 07.
LIDAR_POLICY = "ouster_only"          # same six sensors in every scene, so ground-truth density is comparable
VALIDATE_DOWNLOAD = True              # run the dataset toolkit's own validator on each new block
REVERSE = True                        # backwards here; `03_score` walks the same list forwards,
                                      # so the two servers meet in the middle. Everything else must equal `03_score`.
WORKERS = None                        # scenes scored at once. None = one per CPU core (max 8). The numbers do not depend on it

In [ ]:
# === CODE SYNC (auto-generated by `python -m vggt_aura.sync`, do not edit) ===
raise RuntimeError("The sync cell is empty. On your own machine, in the project folder, run:  python -m vggt_aura.sync   and reopen this notebook.")

In [3]:
# --- 3. Start the session ---
from vggt_aura.session import start_session

# build_cpp=True compiles the C++ geometry core on this server (about 15 s). Ground truth is then built
# with it, which gives exactly the same result as the Python reference, faster. If the build fails,
# everything still runs, in Python.
session = start_session(persist_mode=PERSIST_MODE, drive_root=DRIVE_ROOT, require_gpu=False, build_cpp=True)

Mounted at /content/drive
installing vggt_omega
installing pybind11
installing fzi_aura
persist root: /content/drive/MyDrive/vggt-omega-aura-benchmark
data root   : /content/data/fzi-aura (runtime disk, wiped at session end)
$ cmake -S /content/vggt-omega-aura-benchmark/cpp -B /content/vggt-omega-aura-benchmark/cpp/build -DCMAKE_BUILD_TYPE=Release -Dpybind11_DIR=/usr/local/lib/python3.13/dist-packages/pybind11/share/cmake/pybind11 -DPython_EXECUTABLE=/usr/bin/python3
$ cmake --build /content/vggt-omega-aura-benchmark/cpp/build --config Release -j
C++ core    : built


In [4]:
# --- 4. Process the blocks ---
import pandas as pd
from vggt_aura import aura_data as ad, pipeline as pl

pd.set_option("display.width", 220)
chunks, scene_blocks, hub_files = ad.fetch_release_tables(session.data_root / "_release_tables")
EXCLUDED = ad.fetch_excluded_scene_ids(session.data_root / "_release_tables")   # faulty scenes the maintainers exclude
print("scenes excluded by the dataset:", len(EXCLUDED))
available = ad.available_blocks(chunks, scene_blocks, hub_files, [pl.CAMERA_LAYER, pl.LIDAR_LAYER])
import uuid
ME = ("backward-" if REVERSE else "forward-") + uuid.uuid4().hex[:6]      # this server's name on its claims
summaries, left_to_the_other = [], []
for split, block in (list(reversed(BLOCKS)) if REVERSE else list(BLOCKS)):
    if not pl.block_is_done(session.persist_root, RUN_TAG, MODELS, split, block) \
            and not pl.claim_block(session.persist_root, RUN_TAG, split, block, ME, max_age_s=1500):
        print(f"=== {pl.block_tag(split, block)}: the other server is on it, skipped ===")
        left_to_the_other.append((split, block))
        continue
    assert (split, block) in available.index, f"block {(split, block)} is not downloadable with camera + LiDAR"
    print(f"=== {pl.block_tag(split, block)} ({available.loc[(split, block), 'total_gb']} GB) ===")
    try:
        summary = pl.process_block(session, split, block, ad.block_scene_ids(scene_blocks, split, block, EXCLUDED), CAMERA, MODELS,
                                   RUN_TAG, lidar_policy=LIDAR_POLICY, validate=VALIDATE_DOWNLOAD,
                                   scene_names=ad.block_scene_names(scene_blocks, split, block, EXCLUDED), workers=WORKERS)
    finally:                 # a claim must not outlive a crash: a re-run gets a new name and would wait for it
        pl.release_claim(session.persist_root, RUN_TAG, split, block, ME)
    print(" ", summary)
    summaries.append(summary)
print()
print(pd.DataFrame([{k: v for k, v in s.items() if k not in ("sensors", "validation")} for s in summaries]).to_string(index=False))
still_open = [b for b in left_to_the_other if not pl.block_is_done(session.persist_root, RUN_TAG, MODELS, *b)]
if still_open:
    print()
    print("left to the other server and not finished yet:", still_open, "| if that server stopped, run this notebook again")

scenes excluded by the dataset: 8
=== val_block000012 (2.67 GB) ===
  downloading with the toolkit, decompressing with xz on all 8 cores


  fast unpack: {'archives': 3, 'xz_decompressed_on_all_cores': 1, 'download_s': 7.3, 'verify_and_decompress_s': 33.6, 'extract_s': 10.4}
  validator: {'ok': True, 'scenes_checked': 7, 'errors': []}
  predictions: 0 made now, the rest loaded (20 s) | scoring 7 scenes with 7 worker(s)
  2025-06-13-07-09-37|75      51.4 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2025-06-11-12-27-55|206     56.6 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2025-06-13-07-09-37|74      50.3 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2025-06-13-07-09-37|73      48.5 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2025-06-11-14-31-00|34      73.2 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2025-06-11-14-31-00|37      64.7 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2025-06-16-12-35-26|19      64.0 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  

  fast unpack: {'archives': 4, 'xz_decompressed_on_all_cores': 2, 'download_s': 29.7, 'verify_and_decompress_s': 170.4, 'extract_s': 60.6}
  validator: {'ok': True, 'scenes_checked': 20, 'errors': []}
  predictions: 0 made now, the rest loaded (58 s) | scoring 20 scenes with 8 worker(s)
  2026-01-08-15-27-06|43      26.0 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2026-06-02-17-05-20|48      51.6 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2026-06-02-20-38-47|47      48.6 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2026-06-02-20-38-47|33      60.2 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2026-05-28-15-28-52|54      57.6 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2026-06-02-16-36-33|24      50.9 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2026-06-03-10-44-05|41      49.1 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp

  fast unpack: {'archives': 4, 'xz_decompressed_on_all_cores': 2, 'download_s': 34.2, 'verify_and_decompress_s': 198.0, 'extract_s': 78.4}
  validator: {'ok': True, 'scenes_checked': 20, 'errors': []}
  predictions: 0 made now, the rest loaded (60 s) | scoring 20 scenes with 8 worker(s)
  2026-06-03-10-44-05|40      58.7 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2026-06-02-16-36-33|30      53.2 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2026-06-03-09-57-03|6       50.1 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2026-06-02-20-38-47|39      58.7 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2026-06-02-16-36-33|29      54.6 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2026-06-02-16-36-33|32      51.0 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2026-06-02-16-36-33|25      50.8 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp

  fast unpack: {'archives': 5, 'xz_decompressed_on_all_cores': 3, 'download_s': 47.6, 'verify_and_decompress_s': 297.6, 'extract_s': 139.9}
  validator: {'ok': True, 'scenes_checked': 20, 'errors': []}
  predictions: 0 made now, the rest loaded (59 s) | scoring 20 scenes with 8 worker(s)
  2026-06-02-20-38-47|40      59.4 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2026-06-02-17-05-20|56      60.3 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2026-05-28-16-26-51|43      51.9 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2026-05-28-15-28-52|148     73.6 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2026-06-02-20-38-47|17      55.8 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2026-06-02-17-05-20|127     48.1 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2026-06-02-17-05-20|94      62.5 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cp

  fast unpack: {'archives': 5, 'xz_decompressed_on_all_cores': 3, 'download_s': 44.9, 'verify_and_decompress_s': 291.1, 'extract_s': 206.9}
  validator: {'ok': True, 'scenes_checked': 20, 'errors': []}
  predictions: 0 made now, the rest loaded (60 s) | scoring 20 scenes with 8 worker(s)
  2026-05-28-15-28-52|105    111.4 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2026-06-02-15-30-32|23      96.8 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2026-06-02-15-30-32|39      94.5 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2026-05-28-15-28-52|58      98.0 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2026-05-28-15-28-52|60      88.5 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2026-06-03-10-44-05|58      94.3 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2026-06-02-17-05-20|53      91.9 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cp

In [5]:
# --- 5. What the run holds so far ---
for model in MODELS:
    rows, scenes = pl.load_run(session.persist_root, RUN_TAG, model)
    print(f"{model}: {scenes['scene_id'].nunique() if len(scenes) else 0} scenes in "
          f"{scenes[['split', 'block']].drop_duplicates().shape[0] if len(scenes) else 0} blocks")

vggt_omega_512: 307 scenes in 16 blocks
vggt_1b: 307 scenes in 16 blocks
